# Modern Makine Öğrenmesi Mimarisi Geliştirme
## Hafta 1 — Notebook'tan Production'a

> **Ana fikir:** Notebook kötü bir araç değildir. Keşif ve iletişim için çok güçlüdür; ancak tek başına güvenilir bir production sistemi değildir.

**Ders süresi:** 60 dk anlatım + 60 dk uygulama  
**Örnek problem:** ISP müşterilerinde churn riski

### Bu dersin sonunda ulaşmak istediğim kazanımlar
Bu dersin sonunda birlikte:
- Notebook'un güçlü yanlarını ve production sınırlarını açıklayabileceğiz.
- Gizli durum, hücre sırası, sabit yol ve veri sızıntısı risklerini gösterebileceğiz.
- Basit bir notebook kodunu sorumluluklarına göre modüllere ayırabileceğiz.
- Temel Git döngüsünü ve GitHub/Bitbucket ile çalışma mantığını uygulayabileceğiz.

---
### Bugünkü ders akışımız
1. **0–10 dk:** Notebook neden bu kadar kullanışlı?
2. **10–35 dk:** Canlı anti-pattern deneyleri
3. **35–50 dk:** Production beklentileri ve klasör mimarisi
4. **50–60 dk:** Git ve uzak depo çalışma döngüsü


## 1. Açılış senaryosu

Bir notebook'ta churn modeli geliştirdiğimi ve `%92 accuracy` elde ettiğimi düşünelim. Yarın bu modeli 10 milyon müşteri için her gece çalıştırmam isteniyor.

Derse şu soruyla başlayalım: **Modelim gerçekten hazır mı?**

Bunu anlayabilmek için şu soruları birlikte düşünelim:
- Kod farklı bir bilgisayarda çalışıyor mu?
- Aynı girdi her zaman aynı çıktıyı üretiyor mu?
- Hatalı veri geldiğinde ne oluyor?
- Model ve veri sürümü biliniyor mu?
- Kod test edilebilir ve gözlemlenebilir mi?
- Bir ekip arkadaşı güvenle değiştirebilir mi?

> **Model çalışıyor** ile **sistem güvenilir biçimde çalışıyor** aynı şey değildir.

## 2. Notebook hangi işler için doğru araçtır?

Notebook özellikle şu işler için çok değerlidir:
- Keşifsel veri analizi ve hızlı hipotez denemeleri
- Grafik, tablo, kod ve açıklamayı aynı yerde sunma
- Küçük deneyler ve eğitim
- Sonuçların ekip içinde anlatılması

Production sistemi ise başka nitelikler ister: **tekrar üretilebilirlik, otomasyon, test edilebilirlik, sürümleme, hata yönetimi ve gözlemlenebilirlik**.

Notebook'u çöpe atmıyoruz: keşfi notebook'ta yapıp, kararlı mantığı modüllere taşıyoruz.

In [1]:
# Bu hücre ihtiyaç duyulan örnek veriyi kendisi oluşturur.
# Bilgisayarınızda önceden churn.csv bulunması gerekmez.
from pathlib import Path

import pandas as pd

RANDOM_SEED = 42
DEMO_DATA_DIR = Path.cwd() / "demo_data"
DEMO_DATA_PATH = DEMO_DATA_DIR / "churn.csv"
DEMO_DATA_DIR.mkdir(parents=True, exist_ok=True)

sample_customers = pd.DataFrame({
    "customer_id": range(1001, 1013),
    "tenure_months": [2, 48, 7, 36, 1, 60, 14, 8, 25, 42, 4, 19],
    "monthly_fee": [420, 260, 390, 280, None, 240, 350, 410, 300, 270, 450, 320],
    "support_calls_30d": [5, 0, 3, 1, 7, 0, 2, 4, 1, 0, 6, 2],
    "contract_type": ["monthly", "annual", "monthly", "annual", "monthly", "annual",
                      "monthly", "monthly", "annual", "annual", "monthly", "monthly"],
    "churn": [1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0],
})

# Üretilen veriyi gerçek bir CSV gibi kullanabilmek için diske yazıyoruz.
sample_customers.to_csv(DEMO_DATA_PATH, index=False)
print(f"Örnek veri oluşturuldu: {DEMO_DATA_PATH}")
print(f"Satır sayısı: {len(sample_customers)}")
sample_customers

Örnek veri oluşturuldu: c:\Users\SERKAN\Desktop\Modern Makine Öğrenmesi Mimarisi Geliştirme\modern-makine-ogrenmesi-mimarisi-gelistirme-dersi\hafta_1\demo_data\churn.csv
Satır sayısı: 12


,customer_id,tenure_months,monthly_fee,support_calls_30d,contract_type,churn
0,1001,2,420.0,5,monthly,1
1,1002,48,260.0,0,annual,0
2,1003,7,390.0,3,monthly,1
3,1004,36,280.0,1,annual,0
4,1005,1,NaN,7,monthly,1
5,1006,60,240.0,0,annual,0
6,1007,14,350.0,2,monthly,0
7,1008,8,410.0,4,monthly,1
8,1009,25,300.0,1,annual,0
9,1010,42,270.0,0,annual,0


In [2]:
sample_customers

,customer_id,tenure_months,monthly_fee,support_calls_30d,contract_type,churn
0,1001,2,420.0,5,monthly,1
1,1002,48,260.0,0,annual,0
2,1003,7,390.0,3,monthly,1
3,1004,36,280.0,1,annual,0
4,1005,1,NaN,7,monthly,1
5,1006,60,240.0,0,annual,0
6,1007,14,350.0,2,monthly,0
7,1008,8,410.0,4,monthly,1
8,1009,25,300.0,1,annual,0
9,1010,42,270.0,0,annual,0


## 3. Anti-pattern 1 — Gizli durum ve hücre sırası

Notebook çekirdeği bellekte durum tutar. Hücreler yukarıdan aşağı görünse de farklı sırada çalıştırılabilir. Bu nedenle ekranda gördüğümüz kod, sonucu üretmiş olan gerçek yürütme geçmişiyle aynı olmayabilir.

Şimdi aşağıdaki iki hücreyi sırayla çalıştıralım. Ardından ilk hücreyi tekrar çalıştırmadan ikinci hücreyi birkaç kez çalıştırıp sonucu gözlemleyelim.

In [3]:
discount_rate = 0.10
monthly_revenue = 100_000
print("Başlangıç geliri:", monthly_revenue)

Başlangıç geliri: 100000


In [4]:
# Bu hücre idempotent değildir: her çalıştırma global değişkeni yeniden değiştirir.
monthly_revenue = monthly_revenue * (1 - discount_rate)
print("Hesaplanan gelir:", monthly_revenue)

Hesaplanan gelir: 90000.0


**Tartışma:**
- `Restart Kernel and Run All` yaptığınızda sonuç değişiyor mu?
- Bu kod zamanlanmış bir işte her gece güvenle çalışır mı?
- Fonksiyon girdileri ve çıktıları açık olsaydı ne kazanırdık?

Production yaklaşımı: gizli global durum yerine açık girdi/çıktı kullanan küçük fonksiyonlar.

In [15]:
def apply_discount(revenue: float, rate: float) -> float:
    if not 0 <= rate <= 1:
        raise ValueError("rate 0 ile 1 arasında olmalıdır")
    return revenue * (1 - rate)

apply_discount(revenue=100000, rate=1.2)

ValueError: rate 0 ile 1 arasında olmalıdır

## 4. Anti-pattern 2 — Sabit yollar ve ortama bağımlılık

Aşağıdaki gibi bir kod yazarsam bu kod büyük olasılıkla yalnızca benim bilgisayarımda çalışır:

```python
df = pd.read_csv(r"C:\Users\Serkan\Desktop\churn.csv")
```

Sorun yalnız Windows/Linux farkı değildir. Dosyanın konumu, çalışma dizini ve ortam ayarları koda gömülmüştür.

In [17]:
# Daha taşınabilir yaklaşım: yol parametre olarak gelir ve doğrulanır.
def load_csv(path: str | Path) -> pd.DataFrame:
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f"Veri dosyası bulunamadı: {file_path.resolve()}")
    return pd.read_csv(file_path)

print("Aktif çalışma dizini:", Path.cwd())

# Bir önceki bölümde notebook tarafından oluşturulan CSV'yi okuyoruz.
customers = load_csv(DEMO_DATA_PATH)
print(f"CSV başarıyla okundu: {customers.shape[0]} satır, {customers.shape[1]} kolon")
customers.head()

Aktif çalışma dizini: c:\Users\SERKAN\Desktop\Modern Makine Öğrenmesi Mimarisi Geliştirme\modern-makine-ogrenmesi-mimarisi-gelistirme-dersi\hafta_1


FileNotFoundError: Veri dosyası bulunamadı: C:\Users\SERKAN\Desktop\Modern Makine Öğrenmesi Mimarisi Geliştirme\modern-makine-ogrenmesi-mimarisi-gelistirme-dersi\hafta_1\demo_data\churn.csv

## 5. Anti-pattern 3 — Sessiz veri kalitesi sorunları

Notebook'ta kodun hata vermemesi, verinin doğru olduğu anlamına gelmez. Production girişinde kolon eksik olabilir, veri tipi değişebilir veya beklenmeyen değerler gelebilir.

In [18]:
REQUIRED_COLUMNS = {
    "customer_id", "tenure_months", "monthly_fee",
    "support_calls_30d", "contract_type", "churn"
}

def validate_input(df: pd.DataFrame) -> None:
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"Eksik kolonlar: {sorted(missing)}")
    if df.empty:
        raise ValueError("Veri seti boş olamaz")
    if not df["churn"].dropna().isin([0, 1]).all():
        raise ValueError("churn yalnızca 0 veya 1 olabilir")

validate_input(customers)
print("Temel kontroller başarılı.")

Temel kontroller başarılı.


In [19]:
# Kontrollü hata deneyi: bir zorunlu kolonu kaldırıyoruz.
broken_data = customers.drop(columns=["contract_type"])
try:
    validate_input(broken_data)
except ValueError as exc:
    print("Beklenen doğrulama hatası ->", exc)

Beklenen doğrulama hatası -> Eksik kolonlar: ['contract_type']


## 6. Anti-pattern 4 — Tek hücrede bütün pipeline

Veri okuma, temizleme, feature üretimi, model eğitimi ve dosyaya kaydetme tek hücrede olduğunda:
- Parçaları ayrı ayrı test etmek zorlaşır.
- Aynı preprocessing mantığını servis tarafında yeniden yazma riski oluşur.
- Hatanın hangi aşamada olduğunu bulmak zorlaşır.
- Kod tekrar kullanılamaz hale gelir.

Şimdi küçük bir örnek üzerinden sorumlulukları nasıl ayırabileceğimize bakalım.

In [20]:
FEATURE_COLUMNS = ["tenure_months", "monthly_fee", "support_calls_30d"]
TARGET_COLUMN = "churn"

def prepare_features(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    validate_input(df)
    clean_df = df.copy()
    clean_df["monthly_fee"] = clean_df["monthly_fee"].fillna(
        clean_df["monthly_fee"].median()
    )
    return clean_df[FEATURE_COLUMNS], clean_df[TARGET_COLUMN]

X, y = prepare_features(customers)
print("X shape:", X.shape, "| y shape:", y.shape)
X.head()

X shape: (12, 3) | y shape: (12,)


,tenure_months,monthly_fee,support_calls_30d
0,2,420.0,5
1,48,260.0,0
2,7,390.0,3
3,36,280.0,1
4,1,320.0,7


## 7. Anti-pattern 5 — Tekrar üretilemeyen deney

Rastgelelik kontrol edilmezse aynı kod farklı bölme ve sonuç üretebilir. Bu yalnızca `random_state` meselesi değildir; tekrar üretilebilir bir deney şu bileşenleri birlikte izler:

| Bileşen | Örnek |
|---|---|
| Kod | Git commit kimliği |
| Veri | Veri tarihi/sürümü |
| Konfigürasyon | Feature listesi, model parametreleri |
| Bağımlılıklar | Python ve paket sürümleri |
| Rastgelelik | Seed değerleri |

Notebook'ta hızlıca denenen ayarlar daha sonra `config.yaml`, komut satırı argümanı veya ortam değişkeniyle yönetilebilir.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=RANDOM_SEED, stratify=y
)
model = LogisticRegression(max_iter=1_000)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print("Demo accuracy:", round(accuracy_score(y_test, predictions), 3))
print("Not: Bu küçük yapay veri performans değerlendirmesi için kullanılmıyor.")

Demo accuracy: 0.5
Not: Bu küçük yapay veri performans değerlendirmesi için kullanılmıyor.


## 8. Notebook → production dönüşümü

| Notebook'taki ihtiyaç | Production karşılığı |
|---|---|
| Hızlı keşif | Notebook (`notebooks/`) |
| Tekrarlanan veri okuma | `src/data_loader.py` |
| Feature/temizleme mantığı | `src/preprocessor.py` |
| Eğitim akışı | `src/train.py` |
| Ayarlar | `configs/config.yaml` |
| Güvenilirlik | `tests/` |
| İzlenebilirlik | logging + deney takibi |
| Ortamın tekrar kurulması | `requirements.txt` / `pyproject.toml` |

### Derste kullanacağım başlangıç yapısı

```text
isp-churn-project/
├── README.md
├── .gitignore
├── requirements.txt
├── configs/
│   └── config.yaml
├── data/
│   ├── raw/
│   └── processed/
├── notebooks/
│   └── 01_exploration.ipynb
├── src/
│   ├── __init__.py
│   ├── data_loader.py
│   ├── preprocessor.py
│   └── train.py
├── models/
└── tests/
    └── test_preprocessor.py
```

Bu yapıyı ilk hafta için bilinçli olarak sade tutuyorum. İlerleyen haftalarda üzerine MLflow, API, Docker ve monitoring bileşenlerini birlikte ekleyeceğiz.

## 9. Klasörleri oluşturma — canlı demo

Şimdi aşağıdaki hücreyle `course_workspace/isp-churn-project` altında güvenli bir demo iskeleti oluşturacağım. Bu kod var olan dosyaları silmez.

In [21]:
PROJECT_ROOT = Path.cwd() / "course_workspace" / "isp-churn-project"
DIRECTORIES = [
    "configs", "data/raw", "data/processed", "notebooks",
    "src", "models", "tests"
]

for directory in DIRECTORIES:
    (PROJECT_ROOT / directory).mkdir(parents=True, exist_ok=True)

print(f"Proje iskeleti hazır: {PROJECT_ROOT}")
for path in sorted(PROJECT_ROOT.rglob("*")):
    if path.is_dir():
        print(" -", path.relative_to(PROJECT_ROOT), "/")

Proje iskeleti hazır: c:\Users\SERKAN\Desktop\Modern Makine Öğrenmesi Mimarisi Geliştirme\modern-makine-ogrenmesi-mimarisi-gelistirme-dersi\hafta_1\course_workspace\isp-churn-project
 - configs /
 - data /
 - data\processed /
 - data\raw /
 - models /
 - notebooks /
 - src /
 - tests /


### Dosyaların minimum sorumlulukları

`src/data_loader.py`
```python
from pathlib import Path
import pandas as pd

def load_data(path: str | Path) -> pd.DataFrame:
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(file_path)
    return pd.read_csv(file_path)
```

`src/preprocessor.py`
```python
def prepare_features(df):
    # validation + temizleme + feature seçimi
    return X, y
```

`src/train.py`
```python
def train_model(X, y, random_state=42):
    # eğitim akışı
    return model
```

**Sorumluluk ayrımı:** Veri okuma fonksiyonu model eğitmez; preprocessing fonksiyonu diske model yazmaz.

## 10. Git nedir, GitHub/Bitbucket nedir?

- **Git:** Bilgisayarda çalışan dağıtık sürüm kontrol sistemi.
- **GitHub / Bitbucket:** Git depolarını barındıran; code review, pull request, issue ve CI/CD gibi ekip özellikleri sunan platformlar.

> GitHub veya Bitbucket kullanmak, Git kullanmanın yerine geçmez. Uzak platform, yerel Git geçmişinin ekipçe paylaşılmasını sağlar.

### İlk kurulum (bilgisayar başına bir kez)
```bash
git config --global user.name "Ad Soyad"
git config --global user.email "mail@example.com"
git config --global init.defaultBranch main
git config --list
```

### Yeni yerel depo
```bash
cd isp-churn-project
git init 
git status
git add .
git commit -m "feat: create initial project structure"
```

Notebook hücresinde `!git ...` çalıştırabilirim; fakat gerçek çalışma biçimini daha doğru göstermek için bu bölümde terminal kullanacağım.

## 11. `.gitignore` neden önemlidir?

Büyük veri, model çıktıları, sanal ortamlar ve gizli bilgiler Git'e gönderilmemelidir. Örnek:

```gitignore
# Python
__pycache__/
*.py[cod]
.venv/

# Notebook
.ipynb_checkpoints/

# Veri ve model çıktıları
data/raw/*
data/processed/*
models/*
!data/raw/.gitkeep
!data/processed/.gitkeep
!models/.gitkeep

# Secrets
.env
*.pem
```

> Parola, token ve müşteri verisi yanlışlıkla commit edilirse yalnız son commit'ten silmek yeterli olmayabilir; Git geçmişinde kalabilir. Bu yüzden commit öncesi `git status` ve `git diff --staged` kontrolü alışkanlık olmalıdır.

## 12. GitHub/Bitbucket ile uzak depo akışı

GitHub veya Bitbucket üzerinde boş bir repository oluşturduktan sonra ekranda verilen URL'yi kullanacağım. HTTPS ve SSH'nin iki yaygın bağlantı seçeneği olduğunu burada kısaca göstereceğim.

```bash
# GitHub örneği
git remote add origin https://github.com/KULLANICI/isp-churn-project.git

# Bitbucket örneği — mantık aynıdır
# git remote add origin https://bitbucket.org/WORKSPACE/isp-churn-project.git

git remote -v
git push -u origin main
```

Günlük ekip akışı:
```bash
git switch main
git pull --ff-only
git switch -c feature/data-validation

# Dosyaları değiştir...
git status
git diff
git add src/preprocessor.py tests/test_preprocessor.py
git diff --staged
git commit -m "feat: validate churn input schema"
git push -u origin feature/data-validation
```

Bu değişiklikleri gönderdikten sonra GitHub veya Bitbucket üzerinde bir **Pull Request** açacağım. Ekip incelemesinin ardından değişiklikleri `main` branch'iyle birleştireceğiz.

## 13. Git çalışma modelini okuyalım

```text
Working directory --git add--> Staging area --git commit--> Local repository
Local repository --git push--> GitHub / Bitbucket
GitHub / Bitbucket --git pull--> Local repository + working directory
```

| Komut | Sorduğu soru |
|---|---|
| `git status` | Şu anda ne değişti? |
| `git diff` | Henüz stage etmediğim fark ne? |
| `git add` | Bir sonraki commit'e ne girecek? |
| `git diff --staged` | Commit etmeye hazır fark ne? |
| `git commit` | Yerel geçmişe hangi anlamlı değişikliği kaydediyorum? |
| `git push` | Yerel commit'leri uzak depoya gönder |
| `git pull` | Uzak değişiklikleri getir ve birleştir |

**Kritik alışkanlık:** `git add .` komutunu körlemesine çalıştırmak yerine önce `git status`, sonra mümkünse dosya bazlı `git add` kullanın.

## 14. Bu haftaki kontrol listem ve kapsam sınırım

Bu hafta öğrencilerimin şu temel fikirleri öğrenmesini hedefliyorum:
- Notebook'un keşif için doğru, tek başına production için yetersiz olduğu
- Hücre sırası ve gizli durumun tekrar üretilebilirliği bozabileceği
- Kodun küçük, açık sorumluluklu modüllere ayrılması
- Git'in yerel; GitHub/Bitbucket'ın uzak iş birliği platformu olduğu
- `status → diff → add → commit → push → pull request` temel akışı

Dersin kapsamını yönetebilmek için bu hafta şu konuların detayına girmeyeceğim:
- İleri Git: rebase, cherry-pick, reflog, submodule
- Ayrıntılı branching stratejileri
- CI/CD kurulumu
- DVC/MLflow, Docker ve deployment
- Production düzeyi veri sözleşmesi araçları

Bu konuları yalnızca sonraki haftalara köprü kuracak kadar anacağım.

## 15. Kapanış mini değerlendirmesi

1. Notebook'u production için riskli yapan üç özellik nedir?
2. `Restart Kernel and Run All` neden faydalı bir kontroldür?
3. `src/`, `tests/`, `configs/` klasörlerinin sorumlulukları nelerdir?
4. Git ile GitHub/Bitbucket arasındaki fark nedir?
5. `git add` ile `git commit` arasındaki fark nedir?
6. Müşteri verisi ve `.env` neden Git'e eklenmemelidir?

### Tek cümlelik ders özeti
> Notebook'ta keşfederiz; güvenilir ve tekrar çalıştırılabilir mantığı test edilebilir modüllere taşıyıp Git ile yönetiriz.